In [11]:
import polars as pl
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

# Load data
data = pl.read_parquet(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\data\processed\features.parquet"
)

df = data.to_pandas()

# Filter one store
df = df[df["store_id"] == "CA_1"].copy()

# Keep original values
df_original = df.copy()

# Encode categorical columns
categorical_cols = [
    "store_id",
    "item_id",
    "dept_id",
    "cat_id",
    "weekday"
]

encoder = LabelEncoder()

for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col].astype(str))

# Load trained model
lgbm = joblib.load(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\models\lightgbm_model.pkl"
)

# Get latest record for every item
latest = (
    df.sort_values("date")
      .groupby("item_id")
      .tail(1)
      .reset_index(drop=True)
)

latest_original = (
    df_original.sort_values("date")
               .groupby("item_id")
               .tail(1)
               .reset_index(drop=True)
)

# Features used for prediction
features = [
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "is_weekend",
    "is_event",
    "sell_price",
    "price_change_pct",
    "month",
    "year",
    "weekday",
    "store_id",
    "cat_id",
    "dept_id"
]

X_future = latest[features]

# Forecast
forecast_sales = lgbm.predict(X_future)

# Final Forecast Table
forecast = latest_original[["item_id", "store_id"]].copy()
forecast["forecast_sales"] = forecast_sales

forecast.head(20)

,item_id,store_id,forecast_sales
0,HOUSEHOLD_2_255,CA_1,0.453138
1,HOUSEHOLD_1_277,CA_1,0.140456
2,FOODS_3_179,CA_1,1.253311
3,HOUSEHOLD_2_242,CA_1,0.131593
4,HOUSEHOLD_2_008,CA_1,0.555120
5,FOODS_2_046,CA_1,0.620146
6,FOODS_2_034,CA_1,0.941010
7,HOBBIES_1_273,CA_1,0.470322
8,HOBBIES_2_085,CA_1,0.500037
9,FOODS_3_149,CA_1,0.843183


In [16]:
forecast.to_csv(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\reports\forecast_results.csv",
    index=False
)

Conclusion:


The trained LightGBM model was used to estimate future demand for each product based on the latest available observations. These forecasts serve as the foundation for inventory planning and stock optimization.